In [ ]:
#AI trading agent

In [ ]:
pip install langchain fredapi

In [36]:
#import packs
# Import relevant functionality
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
import getpass
import os
import yfinance as yf
from datetime import datetime, timedelta
import pandas as pd
from fredapi import Fred
import numpy as np

In [2]:
#API Keys
# langsmith API key: 
# fred API key: dcafd942053c4608385fcc4631118e17

In [29]:
fredapikey = getpass.getpass()

In [5]:
#get API keys
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

In [ ]:
#define search tool
search = TavilySearch(max_results=2)

In [21]:
#previous day info
def fetch_stock_ohlc(symbol: str) -> dict:
    try:
        # Download the latest daily data (last trading day)
        stock_data = yf.download(symbol, period="1d", progress=False)
        if stock_data.empty:
            return {"error": f"No data found for symbol '{symbol}'."}
        latest_data = stock_data.iloc[-1]
        return {
            "open": latest_data["Open"],
            "high": latest_data["High"],
            "low": latest_data["Low"],
            "close": latest_data["Close"]
        }
    except Exception as e:
        return {"error": str(e)}

In [24]:
#test 
info = fetch_stock_ohlc('CLPT')
print(info)

{'open': Ticker
CLPT    19.01
Name: 2025-09-26 00:00:00, dtype: float64, 'high': Ticker
CLPT    21.76
Name: 2025-09-26 00:00:00, dtype: float64, 'low': Ticker
CLPT    18.84
Name: 2025-09-26 00:00:00, dtype: float64, 'close': Ticker
CLPT    21.549999
Name: 2025-09-26 00:00:00, dtype: float64}


/var/folders/fm/v1g0kykj4m771hn_zc7_n8q40000gn/T/ipykernel_39211/3881608664.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(symbol, period="1d", progress=False)


In [ ]:
#define tools
tools = [search]

In [41]:
transformed_data = transform_ohlc_data(info)
print(transformed_data)

{'open': np.float64(19.010000228881836), 'high': np.float64(21.760000228881836), 'low': np.float64(18.84000015258789), 'close': np.float64(21.549999237060547)}


In [43]:
def transform_ohlc_data1(ohlc_data: dict, decimals: int = 4) -> dict:
    """
    Transforms a dictionary of pandas Series containing OHLC data into a dictionary
    with only the numerical values for the specified ticker, converted to Python float
    and rounded to the specified number of decimal places.

    Parameters:
    - ohlc_data: dict with keys 'open', 'high', 'low', 'close', each containing a pandas Series
                 with the stock ticker as index and the value for the previous day.
    - decimals: int, number of decimal places to round to (default: 4, suitable for financial data).

    Returns:
    - dict with keys 'open', 'high', 'low', 'close' and their respective numerical values as Python floats.
    """
    ticker = 'CLPT'  # Hardcoded based on provided data; could be dynamic if needed
    return {key: round(float(ohlc_data[key][ticker]), decimals) for key in ohlc_data}

In [45]:
transformed_data1 = transform_ohlc_data1(info)
print(transformed_data1)

{'open': 19.01, 'high': 21.76, 'low': 18.84, 'close': 21.55}


In [49]:
def set_trading_thresholds(ohlc_data: dict) -> dict:
    """
    Calculates dynamic trading thresholds and limits for a Doji-based strategy using OHLC data.

    Parameters:
    - ohlc_data: dict with keys 'open', 'high', 'low', 'close' containing numerical values (floats).

    Returns:
    - dict with calculated thresholds: 'stop_loss', 'take_profit', 'entry_threshold', 'volatility'
    
    Assumptions:
    - Uses high-low range as a simple volatility proxy (approximating ATR).
    - Stop-loss: close - (volatility * risk_factor), where risk_factor=1.5
    - Take-profit: close + (volatility * reward_factor), where reward_factor=2.0
    - Entry threshold: small percentage of close for Doji confirmation (e.g., 0.1%)
    """
    # Extract numerical values directly from the dictionary
    open_price = ohlc_data['open']
    high_price = ohlc_data['high']
    low_price = ohlc_data['low']
    close_price = ohlc_data['close']
    
    # Calculate volatility (high - low range as simple ATR proxy)
    volatility = high_price - low_price
    
    # Define risk and reward factors (customizable)
    risk_factor = 1.5  # Multiplier for stop-loss
    reward_factor = 2.0  # Multiplier for take-profit
    entry_buffer = 0.005  # 0.1% of close for Doji body size threshold
    
    # Calculate thresholds
    # Calculate thresholds
    stop_loss = close_price - (volatility * risk_factor)
    take_profit = close_price + (volatility * reward_factor)
    bullish_entry_price = high_price + (volatility * entry_buffer)  # Entry for buy above high
    bearish_entry_price = low_price - (volatility * entry_buffer)  # Entry for sell below low
    
    return {
        'ticker': 'CLPT',  # Hardcoded based on context; could be dynamic if needed
        'volatility': volatility,
        'stop_loss': stop_loss,
        'take_profit': take_profit,
        'bullish_entry_price': bullish_entry_price,
        'bearish_entry_price': bearish_entry_price
    }

In [50]:
limits1 = set_trading_thresholds(transformed_data1)
print(limits1)

{'ticker': 'CLPT', 'volatility': 2.9200000000000017, 'stop_loss': 17.169999999999998, 'take_profit': 27.390000000000004, 'bullish_entry_price': 21.774600000000003, 'bearish_entry_price': 18.8254}
